<a href="https://colab.research.google.com/github/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

My baseline rule (w04) and my model (w05/w06) each know something the other doesn't, so the
queue combines them rather than picking one:

- **Baseline rule gates** (`stale`, `has_demand`, `weak_ctr`) — simple, already precision-audited:
  precision@50 = 0.80, and precision@19 (its own flagged count) = 1.00 on the w05 holdout.
- **Model risk score** — same Random Forest as w05/w06, but scored differently here. w05/w06 only
  ever showed me probabilities for ONE 70/30 grouped test split. For a queue that has to rank the
  *whole* 30k-row inventory, I need an honest score for every row, so I use `GroupKFold(n_splits=5)`
  by `client_id`: 5 models, each scoring only clients it never trained on, pooled into one
  out-of-fold (OOF) probability per row — no row is ever scored by a model that saw its own client.

**A finding that changes my design:** the 5 fold AUCs came back 0.65 / 0.63 / 0.73 / 0.65 / 0.67
(mean 0.667) — well below the single grouped split's 0.763 from w06. With only 32 clients total,
one grouped 70/30 split is a small, noisy sample of "which clients land in test," so the earlier
0.763 was likely an optimistic draw, not a stable number. **Because of this, I do NOT let the raw
model probability drive the primary ranking** — the rule gates (more stable, already audited) set
the main tier, and the model score is used only as (a) a way to catch "recently refreshed but the
model still reads it as high-risk on age/traffic alone" — exactly the w05 error-analysis pattern —
and (b) a tie-break sort inside tiers with mixed signal.

**Six action tiers (rule-based segments, not a fitted clustering model — I have not run K-Means or
any clustering here, so I don't call these "clusters"):**

| Tier | Reason code | Action | What it means |
|---|---|---|---|
| 1 | `stale_weak_ctr_with_demand` | `refresh_priority` | Stale (90+ days), real traffic, CTR below its position tier's median — the w04 rule, unchanged |
| 2 | `weak_ctr_no_stale_ctr_fix` | `ctr_fix` | Weak CTR + real traffic, but NOT stale — a lighter title/meta fix, not a full rewrite |
| 3 | `recent_refresh_suppress_high_risk_score` | `monitor_recent_refresh` | Updated in the last 20 days AND the model still scores it high-risk — the w05 error pattern; watch, don't re-flag |
| 4 | `protect_stable_performer` | `protect` | Real traffic, CTR not weak, model risk score in the bottom third — leave alone |
| 5 | `low_demand_deprioritize` | `deprioritize_low_demand` | Under 500 impressions/90d — not enough audience to justify review time regardless of other signals |
| 6 | `no_position_data_exclude` | `exclude_no_data` | `avg_position == 0` (no data, per the data dictionary) — can't judge CTR-vs-position for these at all |
| — | `mixed_signal_monitor` | `monitor_general` | Doesn't cleanly match any gate above (e.g. stale but average CTR, or fresh with a mid risk score) — worth a periodic look, not urgent |

The `declined` column below is `trend_direction == "down"` used only as a **post-hoc sanity check**
on the tiers (never as a rule or model input) — same label-leakage rule as every earlier notebook.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii"
REPO_DIR = "flyrank-ml-internship-hadii"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import json
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 1

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Loaded:", df.shape)

label = (df["trend_direction"] == "down").astype(int)

work = df.copy()
work["has_search_volume_data"] = work["search_volume"].notna().astype(int)
work["has_word_count_data"] = work["word_count"].notna().astype(int)
work["has_position_data_flag"] = (work["avg_position"] > 0).astype(int)

# Same honest, leakage-checked feature set as w05_model / w06_validation_audit (38 columns).
numeric_candidates = [
    "content_age_days", "days_since_last_update", "word_count", "char_count",
    "search_volume", "competition", "cpc",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
X_numeric = work[numeric_candidates].fillna(0)
X_numeric = pd.concat(
    [X_numeric, work[["has_search_volume_data", "has_word_count_data", "has_position_data_flag"]]],
    axis=1,
)
categorical_candidates = ["content_type", "main_intent", "competition_level"]
X_categorical = pd.get_dummies(work[categorical_candidates], dummy_na=True, prefix=categorical_candidates)
X = pd.concat([X_numeric, X_categorical], axis=1)
groups = work["client_id"]
print("Feature matrix:", X.shape, "| clients:", groups.nunique())

# --- Honest out-of-fold model risk score for the WHOLE inventory ---
gkf = GroupKFold(n_splits=5)
oof_proba = np.zeros(len(X))
fold_aucs = []
for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, label, groups)):
    rf = RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=20,
        random_state=RANDOM_SEED, n_jobs=-1,
    )
    rf.fit(X.iloc[tr_idx], label.iloc[tr_idx])
    proba = rf.predict_proba(X.iloc[te_idx])[:, 1]
    oof_proba[te_idx] = proba
    fold_auc = roc_auc_score(label.iloc[te_idx], proba)
    fold_aucs.append(fold_auc)
    print(f"Fold {fold}: train clients={groups.iloc[tr_idx].nunique():>2}, "
          f"test clients={groups.iloc[te_idx].nunique():>2}, AUC={fold_auc:.3f}")

print(f"\nFold AUC spread: min={min(fold_aucs):.3f}, mean={np.mean(fold_aucs):.3f}, "
      f"max={max(fold_aucs):.3f}  (compare to w06's single-split grouped AUC: 0.763)")

work["model_risk_score"] = oof_proba
q1, q2 = work["model_risk_score"].quantile([1 / 3, 2 / 3])
work["model_risk_tier"] = pd.cut(
    work["model_risk_score"], bins=[-1, q1, q2, 2], labels=["low", "mid", "high"]
)

# --- Rule gates (unchanged from w04) ---
clean = df[df["avg_position"] > 0]
tier_median_ctr = clean.groupby("position_tier", observed=True)["ctr"].median()
work["tier_median_ctr"] = work["position_tier"].map(tier_median_ctr)
work["stale"] = work["days_since_last_update"] >= 90
work["has_demand"] = work["impressions_90d"] >= 500
work["weak_ctr"] = (work["has_position_data_flag"] == 1) & (work["ctr"] < work["tier_median_ctr"])
work["recent_refresh"] = work["days_since_last_update"] <= 20

# --- Six tiers, evaluated in priority order (first match wins) ---
conditions = [
    work["has_position_data_flag"] == 0,
    (~work["has_demand"]),
    work["stale"] & work["has_demand"] & work["weak_ctr"],
    work["recent_refresh"] & (work["model_risk_tier"] == "high"),
    (~work["stale"]) & work["has_demand"] & work["weak_ctr"],
    work["has_demand"] & (~work["weak_ctr"]) & (work["model_risk_tier"] == "low"),
]
actions = ["exclude_no_data", "deprioritize_low_demand", "refresh_priority",
           "monitor_recent_refresh", "ctr_fix", "protect"]
reason_codes = [
    "no_position_data_exclude", "low_demand_deprioritize", "stale_weak_ctr_with_demand",
    "recent_refresh_suppress_high_risk_score", "weak_ctr_no_stale_ctr_fix",
    "protect_stable_performer",
]
work["action"] = np.select(conditions, actions, default="monitor_general")
work["reason_code"] = np.select(conditions, reason_codes, default="mixed_signal_monitor")

# Rank score: baseline-style (impressions-weighted) inside refresh/ctr-fix tiers,
# model risk score inside the monitor tiers, impressions as a stable tie-break elsewhere.
tier_priority = {
    "refresh_priority": 1, "ctr_fix": 2, "monitor_recent_refresh": 3,
    "monitor_general": 4, "protect": 5, "deprioritize_low_demand": 6, "exclude_no_data": 7,
}
work["tier_priority"] = work["action"].map(tier_priority)
work["sort_score"] = np.where(
    work["action"].isin(["refresh_priority", "ctr_fix"]), work["impressions_90d"],
    np.where(work["action"].isin(["monitor_recent_refresh", "monitor_general"]),
             work["model_risk_score"] * work["impressions_90d"].clip(lower=1),
             work["impressions_90d"]),
)

queue = (
    work.sort_values(["tier_priority", "sort_score"], ascending=[True, False])
        .reset_index(drop=True)
)
queue.insert(0, "rank", queue.index + 1)

print("\nAction tier counts:")
print(queue["action"].value_counts())
print("\nSanity check: total rows across tiers:", len(queue), "(should be 30000)")

# --- Sanity check (post-hoc only): observed decline rate by tier ---
queue["declined"] = (df.set_index(df["content_id"]).loc[queue["content_id"], "trend_direction"] == "down").values
print("\nObserved decline rate by action tier (directional check, NOT a model input):")
print(queue.groupby("action")["declined"].mean().sort_values(ascending=False).round(3))

display_cols = ["rank", "content_id", "client_id", "action", "reason_code",
                 "model_risk_score", "impressions_90d", "ctr", "position_tier",
                 "days_since_last_update"]
queue[display_cols].head(10)

Loaded: (30000, 44)
Feature matrix: (30000, 38) | clients: 32
Fold 0: train clients=31, test clients= 1, AUC=0.653
Fold 1: train clients=25, test clients= 7, AUC=0.630
Fold 2: train clients=24, test clients= 8, AUC=0.732
Fold 3: train clients=24, test clients= 8, AUC=0.654
Fold 4: train clients=24, test clients= 8, AUC=0.667

Fold AUC spread: min=0.630, mean=0.667, max=0.732  (compare to w06's single-split grouped AUC: 0.763)

Action tier counts:
action
deprioritize_low_demand    12069
monitor_general             5494
monitor_recent_refresh      4226
protect                     3469
refresh_priority            2019
ctr_fix                     1518
exclude_no_data             1205
Name: count, dtype: int64

Sanity check: total rows across tiers: 30000 (should be 30000)

Observed decline rate by action tier (directional check, NOT a model input):
action
refresh_priority           0.707
monitor_recent_refresh     0.685
monitor_general            0.584
ctr_fix                    0.543
depr

,rank,content_id,client_id,action,reason_code,model_risk_score,impressions_90d,ctr,position_tier,days_since_last_update
0,1,content_5fe46e04994d,client_4e07408562,refresh_priority,stale_weak_ctr_with_demand,0.425929,517715,0.14,page_1,104
1,2,content_36ff89c8214e,client_19581e27de,refresh_priority,stale_weak_ctr_with_demand,0.427395,295097,0.05,page_1,104
2,3,content_c8e9d6ab9013,client_19581e27de,refresh_priority,stale_weak_ctr_with_demand,0.562457,208678,0.00,page_1,104
3,4,content_a7427266c305,client_19581e27de,refresh_priority,stale_weak_ctr_with_demand,0.412024,201111,0.11,page_1,104
4,5,content_91652435f57a,client_19581e27de,refresh_priority,stale_weak_ctr_with_demand,0.478825,159590,0.06,page_1,104
5,6,content_f42eb861c6dd,client_19581e27de,refresh_priority,stale_weak_ctr_with_demand,0.437532,152467,0.13,page_1,104
6,7,content_11fcfd65d94c,client_19581e27de,refresh_priority,stale_weak_ctr_with_demand,0.520714,149083,0.15,page_1,104
7,8,content_97a86caf3a3d,client_19581e27de,refresh_priority,stale_weak_ctr_with_demand,0.422440,147670,0.07,page_1,104
8,9,content_8b36799b7e44,client_6208ef0f77,refresh_priority,stale_weak_ctr_with_demand,0.553024,141400,0.02,page_3_5,104
9,10,content_c1fe78bc4e37,client_19581e27de,refresh_priority,stale_weak_ctr_with_demand,0.486852,134055,0.03,page_1,104


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who uses this:** a content or SEO team lead running a weekly or biweekly refresh triage on
FlyRank Lane-2-style clients whose data profile resembles the 32 clients in the starter sample —
using this to decide *where to spend limited human review time first*, not to decide the final
action alone.

**What it is:** a priority-ordered reading list with a stated reason for each row. What it is NOT:
a prediction that any specific page's ranking or traffic *will* change, a substitute for reading
the page, or a tool that works the same way for a client type not represented in training.

**Limits, stated plainly:**

1. **Small-client-count generalization risk.** Only 32 clients total; the 5-fold grouped-CV AUC
   swung from 0.63 to 0.73 across folds (Section 1). Any single number this model reports should be
   read as "roughly this good, with real fold-to-fold noise," not a fixed guarantee.
2. **The label is a proxy, not ground truth.** `trend_direction` compares a page's own recent window
   to its own earlier window — the top model features (w05) are age and traffic-history signals, so
   the model partly learns "this page has accumulated enough history to show a measured trend" as
   much as it learns true decline.
3. **One 90-day snapshot, no seasonality.** No visibility into whether a page's dip is seasonal,
   algorithm-update-driven, or a genuine content problem.
4. **The CTR benchmark drifts.** `tier_median_ctr` is computed from this exact dataset; it needs
   periodic recomputation as content mix and SERP behavior change (see Section 4).
5. **Not validated against the ~79M-row warehouse** — only the 30k-row starter sample. Scores on the
   full release would need their own leakage and split audit before trusting them the same way.
6. **Cross-sectional, not causal.** Per the honest-claims ladder: this supports "these pages look
   worth reviewing first, because..." — never "refreshing this page will increase traffic."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Backing numbers for the limits above (same variables as Section 1 — this notebook runs top to bottom).
print("Total clients in dataset:", groups.nunique())
print(f"Fold AUC range: {min(fold_aucs):.3f} - {max(fold_aucs):.3f}  (spread: {max(fold_aucs) - min(fold_aucs):.3f})")
print(f"Fold AUC mean: {np.mean(fold_aucs):.3f}  vs. single grouped-split AUC from w06: 0.763")
print(f"\nLabel base rate (share declining, full 30k rows): {round(label.mean(), 3)}")
print(f"Rows with no position data (avg_position == 0): {(df['avg_position'] == 0).sum():,} "
      f"({(df['avg_position'] == 0).mean():.1%} of inventory)")
print(f"\nCurrent tier-median CTR benchmark (recompute periodically -- see Section 4):")
print(tier_median_ctr.to_string())

Total clients in dataset: 32
Fold AUC range: 0.630 - 0.732  (spread: 0.102)
Fold AUC mean: 0.667  vs. single grouped-split AUC from w06: 0.763

Label base rate (share declining, full 30k rows): 0.542
Rows with no position data (avg_position == 0): 1,205 (4.0% of inventory)

Current tier-median CTR benchmark (recompute periodically -- see Section 4):
position_tier
deep        0.00
page_1      0.16
page_3_5    0.03
striking    0.11
top_3       0.00


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any row, a human must:**

- Open the live page and confirm the reason code still matches reality (a manual edit after this
  snapshot won't show up here).
- Check whether a low CTR is explained by something the data can't see — e.g. the page already
  wins a featured snippet or a non-clicky SERP feature, where "weak_ctr" doesn't mean "fixable by
  a rewrite."
- Confirm the page isn't already scheduled for a redesign, merge, or planned removal that this
  queue has no way of knowing about.
- For `monitor_recent_refresh` rows specifically: check the refresh date against the model risk
  score's own logic — if it's still high-risk after 30-60 more days, that's the point to
  reconsider, not to re-flag it as `refresh_priority` again.

**What should NOT be automated — no-go list:**

- **No automatic publishing.** No title/meta/content change goes live without a human writing or
  approving the actual replacement text.
- **No performance evaluation of writers or editors** based on this queue or its reason codes —
  the label is a traffic-trend proxy, not a judgment about who wrote the page or when.
- **No causal claims to clients.** Never say "the model predicts X% more traffic" — see the
  honest-claims limit in Section 2; this is decision support, not a forecast.
- **No bulk automated action triggered by score alone** — even `refresh_priority`, the tier with
  the strongest rule-audit backing (precision@50 = 0.80 in w05), is a review queue entry, not a
  standing work order.
- **No applying this queue to a client type absent from the 32 in the training data** without
  first re-running the leakage and grouped-CV checks on that client's data.
- **Never treat `monitor_recent_refresh` as "solved and safe to ignore forever."** It exists
  because the model's own error pattern (w05, Section 4) shows it can misread a recently-fixed
  page as still declining — the honest response is to watch it again later, not to drop it.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Concrete count backing the no-go note on monitor_recent_refresh: rows where a very recent edit
# still scores high-risk -- exactly the w05 error pattern, now counted across the whole inventory.
monitor_recent = queue[queue["action"] == "monitor_recent_refresh"]
print(f"monitor_recent_refresh rows: {len(monitor_recent):,} "
      f"({len(monitor_recent) / len(queue):.1%} of inventory)")
print(f"Median days since last update in this tier: {monitor_recent['days_since_last_update'].median():.0f} "
      f"(all <= 20 by construction)")
print(f"Median model risk score in this tier: {monitor_recent['model_risk_score'].median():.2f}")
print("\nThese are exactly the pages a naive score-only automation would wrongly re-flag for another")
print("refresh -- the no-go list exists because of this specific, observed pattern, not a hypothetical.")

monitor_recent_refresh rows: 4,226 (14.1% of inventory)
Median days since last update in this tier: 20 (all <= 20 by construction)
Median model risk score in this tier: 0.68

These are exactly the pages a naive score-only automation would wrongly re-flag for another
refresh -- the no-go list exists because of this specific, observed pattern, not a hypothetical.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Retrain the model when:**

- A new client is onboarded whose data changes the 32-client mix meaningfully (e.g. a very
  different content_type distribution) — the grouped-CV spread (0.63-0.73 AUC) already shows this
  model is sensitive to which clients are in the mix.
- The realized precision@50 on actually-flagged `refresh_priority` rows (tracked over the next
  review cycle, once outcomes are known) drops notably below the w05-audited 0.80-0.84 range.
- The label base rate (currently 0.542) drifts far from that number — a sign the underlying content
  mix or measurement window has shifted.
- The fold-to-fold AUC spread widens further on a re-run — a sign the client-generalization problem
  is getting worse, not better.

**Recompute (lighter than a full retrain) when:**

- `tier_median_ctr` — the CTR benchmark each `weak_ctr` gate depends on — monthly, since SERP
  behavior and content mix shift faster than the model needs retraining.
- The staleness/decline relationship at the 181+ day bucket (w04 flagged this as a reversal, but
  on only n=174 rows, too small to trust) — recheck once more data accumulates in that bucket.

**Monitoring cadence proposed:** recompute the CTR benchmark and re-score the queue monthly;
re-run the full 5-fold grouped-CV retrain-and-compare quarterly, or immediately after any client
onboarding/offboarding.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# A small, saveable checklist of current reference numbers -- what "normal" looks like today,
# so a future run can compare against these and decide if a trigger has actually fired.
monitoring_baseline = {
    "label_base_rate": round(float(label.mean()), 3),
    "fold_auc_min": round(float(min(fold_aucs)), 3),
    "fold_auc_mean": round(float(np.mean(fold_aucs)), 3),
    "fold_auc_max": round(float(max(fold_aucs)), 3),
    "n_clients": int(groups.nunique()),
    "tier_median_ctr": {k: float(v) for k, v in tier_median_ctr.items()},
    "action_tier_counts": queue["action"].value_counts().to_dict(),
}
for k, v in monitoring_baseline.items():
    print(f"{k}: {v}")

label_base_rate: 0.542
fold_auc_min: 0.63
fold_auc_mean: 0.667
fold_auc_max: 0.732
n_clients: 32
tier_median_ctr: {'deep': 0.0, 'page_1': 0.16, 'page_3_5': 0.03, 'striking': 0.11, 'top_3': 0.0}
action_tier_counts: {'deprioritize_low_demand': 12069, 'monitor_general': 5494, 'monitor_recent_refresh': 4226, 'protect': 3469, 'refresh_priority': 2019, 'ctr_fix': 1518, 'exclude_no_data': 1205}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Two files for the paper's recommendations section: the full ranked queue (regenerated by this
notebook, not committed as a CSV per the repo's data leak-guard) and a small metrics JSON (the
receipts — committed, since it's not a data dump). One figure goes to `work/figures/` for reuse.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# --- 1. Ranked queue CSV (stays out of git by design -- work/**/*.csv is gitignored;
#        this notebook regenerates it on demand). ---
export_cols = ["rank", "content_id", "client_id", "action", "reason_code",
               "model_risk_score", "model_risk_tier", "sort_score",
               "stale", "has_demand", "weak_ctr", "recent_refresh",
               "days_since_last_update", "impressions_90d", "ctr",
               "position_tier", "avg_position"]
queue[export_cols].to_csv("work/outputs/w07_action_playbook_queue.csv", index=False)
print("Wrote", len(queue), "rows to work/outputs/w07_action_playbook_queue.csv")

# --- 2. Metrics JSON -- the receipts the paper's numbers trace back to (committed). ---
metrics = {
    "notebook": "w07_action_playbook",
    "model": "RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20)",
    "validation": "GroupKFold(n_splits=5) by client_id, out-of-fold probabilities pooled across all rows",
    "fold_aucs": [round(float(a), 3) for a in fold_aucs],
    "fold_auc_mean": round(float(np.mean(fold_aucs)), 3),
    "single_split_grouped_auc_w06_reference": 0.763,
    "baseline_rule_precision_at_50_w05_reference": 0.80,
    "baseline_rule_precision_at_flagged_count_w05_reference": 1.00,
    "label_base_rate": round(float(label.mean()), 3),
    "n_clients": int(groups.nunique()),
    "action_tier_counts": queue["action"].value_counts().to_dict(),
    "observed_decline_rate_by_tier": queue.groupby("action")["declined"].mean().round(3).to_dict(),
    "tier_median_ctr_benchmark": {k: float(v) for k, v in tier_median_ctr.items()},
}
with open("work/outputs/w07_action_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote work/outputs/w07_action_playbook_metrics.json")

# --- 3. Figure: action tier counts (reused in the paper) ---
fig, ax = plt.subplots(figsize=(8, 4.5))
counts = queue["action"].value_counts().sort_values()
ax.barh(counts.index, counts.values, color="#3b6fa0")
ax.set_xlabel("Rows in tier")
ax.set_title("Content action playbook -- rows per action tier (n=30,000)")
for i, v in enumerate(counts.values):
    ax.text(v + 200, i, f"{v:,}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("work/figures/w07_action_tier_counts.png", dpi=150)
plt.close()
print("Wrote work/figures/w07_action_tier_counts.png")

Wrote 30000 rows to work/outputs/w07_action_playbook_queue.csv
Wrote work/outputs/w07_action_playbook_metrics.json
Wrote work/figures/w07_action_tier_counts.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.